# Provider Comparison Template With OpenAI

This notebook complements `14_model_comparison_scenarios.py`.
It uses several OpenAI models as a practical baseline for a provider-style comparison workflow.

The same rubric can later be reused across other providers, but keeping the notebook inside the OpenAI labs makes it easy to start with one consistent SDK and one billing model.

In [1]:
import ast
import json
import sys
import time
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, ValidationError


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "01_openai" / "scripts").exists():
            return candidate
    raise RuntimeError("Could not find the project root from the current notebook path")


PROJECT_ROOT = find_project_root()
SCRIPTS_DIR = PROJECT_ROOT / "01_openai" / "scripts"

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))

load_dotenv(PROJECT_ROOT / ".env")
client = OpenAI()


## Comparison Setup

We will compare three OpenAI models on the same prompt using four dimensions:

- response quality
- latency
- estimated cost
- JSON reliability


In [2]:
MODELS = ["gpt-5", "gpt-5-mini", "gpt-5-nano"]

PRICING_PER_1M_TOKENS = {
    "gpt-5": {"input": 1.25, "cached_input": 0.125, "output": 10.00},
    "gpt-5-mini": {"input": 0.25, "cached_input": 0.025, "output": 2.00},
    "gpt-5-nano": {"input": 0.05, "cached_input": 0.005, "output": 0.40},
}

PROMPT = """
Answer the following question as valid JSON only.

Question:
Explain the difference between a Python list and a Python tuple, when to use each one,
and provide one very small example.

Return exactly this JSON shape:
{
  "summary": "string",
  "differences": [
    {"topic": "string", "list": "string", "tuple": "string"},
    {"topic": "string", "list": "string", "tuple": "string"},
    {"topic": "string", "list": "string", "tuple": "string"}
  ],
  "example_code": "string",
  "best_choice_rule": "string",
  "common_mistake": "string"
}

Rules:
- Return raw JSON only.
- `differences` must contain exactly 3 items.
- Each `topic` must be one of: `mutability`, `performance`, `use_case`.
- Mention mutability clearly.
""".strip()


In [3]:
class DifferenceItem(BaseModel):
    topic: str
    list: str
    tuple: str


class ComparisonPayload(BaseModel):
    summary: str
    differences: list[DifferenceItem]
    example_code: str
    best_choice_rule: str
    common_mistake: str


def estimate_cost_usd(model: str, usage) -> float:
    pricing = PRICING_PER_1M_TOKENS[model]
    cached_input_tokens = usage.input_tokens_details.cached_tokens
    non_cached_input_tokens = usage.input_tokens - cached_input_tokens

    input_cost = (non_cached_input_tokens / 1_000_000) * pricing["input"]
    cached_input_cost = (cached_input_tokens / 1_000_000) * pricing["cached_input"]
    output_cost = (usage.output_tokens / 1_000_000) * pricing["output"]
    return input_cost + cached_input_cost + output_cost


def parse_payload(raw_output: str) -> ComparisonPayload | None:
    try:
        payload = json.loads(raw_output)
        return ComparisonPayload.model_validate(payload)
    except (json.JSONDecodeError, ValidationError):
        return None


def is_python_example_valid(example_code: str) -> bool:
    try:
        parsed = ast.parse(example_code)
    except SyntaxError:
        return False

    if len(parsed.body) != 2:
        return False

    assignments = [node for node in parsed.body if isinstance(node, ast.Assign)]
    if len(assignments) != 2:
        return False

    values = [node.value for node in assignments]
    has_list = any(isinstance(value, ast.List) for value in values)
    has_tuple = any(isinstance(value, ast.Tuple) for value in values)
    return has_list and has_tuple


def score_quality(payload: ComparisonPayload | None) -> float:
    if payload is None:
        return 0.0

    score = 0.0
    text = json.dumps(payload.model_dump()).lower()

    if len(payload.differences) == 3:
        score += 2.0
    if "mutable" in text or "immutable" in text:
        score += 2.0
    if "use" in text or "when" in text:
        score += 2.0
    if "hashable" in text or "dictionary key" in text:
        score += 2.0
    if is_python_example_valid(payload.example_code):
        score += 2.0

    return score


## Run One Comparison Pass

In [4]:
results = []

for model in MODELS:
    started_at = time.perf_counter()
    response = client.responses.create(
        model=model,
        reasoning={"effort": "minimal"},
        input=PROMPT,
    )
    latency_seconds = time.perf_counter() - started_at
    payload = parse_payload(response.output_text)

    results.append(
        {
            "model": model,
            "latency_seconds": latency_seconds,
            "estimated_cost_usd": estimate_cost_usd(model, response.usage),
            "json_valid": payload is not None,
            "quality_score": score_quality(payload),
            "input_tokens": response.usage.input_tokens,
            "output_tokens": response.usage.output_tokens,
            "raw_output": response.output_text,
        }
    )

results


[{'model': 'gpt-5',
  'latency_seconds': 8.101528709055856,
  'estimated_cost_usd': 0.0032949999999999998,
  'json_valid': True,
  'quality_score': 8.0,
  'input_tokens': 196,
  'output_tokens': 305,
  'raw_output': '{\n  "summary": "Lists are mutable, ordered collections suited for data that changes; tuples are immutable, ordered collections suited for fixed data and lightweight, hashable groupings.",\n  "differences": [\n    {"topic": "mutability", "list": "Mutable: items can be added, removed, or changed in place.", "tuple": "Immutable: contents cannot be changed after creation."},\n    {"topic": "performance", "list": "Slightly slower and larger overhead due to mutability and dynamic resizing.", "tuple": "Slightly faster and smaller; can be used as dict keys if contents are hashable."},\n    {"topic": "use_case", "list": "Use for collections that will change (e.g., accumulating results).", "tuple": "Use for fixed records or return multiple values safely without accidental edits."}\

## Read The Results

This small formatter makes the trade-offs easier to scan in notebook form.

In [5]:
header = (
    "model           quality  json_valid  latency_s  cost_usd    "
    "input_tokens  output_tokens"
)
print(header)
print("-" * len(header))

for item in results:
    print(
        f"{item['model']:<15} "
        f"{item['quality_score']:>7.1f} "
        f"{str(item['json_valid']):>11} "
        f"{item['latency_seconds']:>10.2f} "
        f"{item['estimated_cost_usd']:>10.6f} "
        f"{item['input_tokens']:>13} "
        f"{item['output_tokens']:>14}"
    )


model           quality  json_valid  latency_s  cost_usd    input_tokens  output_tokens
---------------------------------------------------------------------------------------
gpt-5               8.0        True       8.10   0.003295           196            305
gpt-5-mini          6.0        True       8.00   0.000835           196            393
gpt-5-nano          8.0        True       4.67   0.000177           196            417


## Inspect One Raw Output

If one model behaves unexpectedly, inspect its raw JSON directly.

In [6]:
results[0]["raw_output"]


'{\n  "summary": "Lists are mutable, ordered collections suited for data that changes; tuples are immutable, ordered collections suited for fixed data and lightweight, hashable groupings.",\n  "differences": [\n    {"topic": "mutability", "list": "Mutable: items can be added, removed, or changed in place.", "tuple": "Immutable: contents cannot be changed after creation."},\n    {"topic": "performance", "list": "Slightly slower and larger overhead due to mutability and dynamic resizing.", "tuple": "Slightly faster and smaller; can be used as dict keys if contents are hashable."},\n    {"topic": "use_case", "list": "Use for collections that will change (e.g., accumulating results).", "tuple": "Use for fixed records or return multiple values safely without accidental edits."}\n  ],\n  "example_code": "my_list = [1, 2]; my_list.append(3)\\nmy_tuple = (1, 2, 3)  # my_tuple[0] = 9  # TypeError",\n  "best_choice_rule": "If the data should not change after creation, prefer a tuple; otherwise, 

## Practical Takeaways

- Start with one prompt and one evaluation rubric.
- Compare latency, cost, and output quality together rather than separately.
- JSON reliability is often as important as answer quality in real applications.
- This notebook uses OpenAI models only, but the structure is intentionally reusable if you later compare other providers with a compatible evaluation harness.
